# Upgrading to Pretrained Base Models

DustyLM is designed to let you build an 8M parameter model from scratch so you can see every part of the training loop. However, an 8M model is mathematically limited. It simply cannot store deep language knowledge or perform complex reasoning. 

To build a truly capable assistant, the industry standard is to start with a much larger, already-trained open-source model and use Supervised Fine-Tuning (SFT) to align it to your specific persona. 

In this notebook, we will use Hugging Face's **SmolLM2** (135M and 360M parameter versions) as our advanced base models. You will learn how DustyLM profiles manage external weights, how to convert Hugging Face checkpoints into the local format, and how SFT fits into the exact same profile-driven workflow you already know.


## Setup

Run this in Colab, RunPod, or a local notebook. The setup cell auto-detects your environment.

In [ ]:
import sys, os
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
    os.chdir("dusty-lm")
elif not Path("dustylm").exists():
    print("Run this from the repo root: cd /path/to/dusty-lm && uv sync")



## 1. Understanding Profiles in DustyLM

To keep the codebase extremely flexible without requiring you to rewrite the training loop for every new model, this repository uses a central configuration system called **Profiles**. All profiles are defined in the repository's main configuration file and dictate how a model is built, trained, and run.

There are two main categories of profiles:

*   **From-Scratch Profiles (e.g., `dusty8m`, `sft_dusty8m`):** These define custom, lightweight architectures. You used these in the previous notebooks to build and train your own tiny model from the ground up. You have total freedom to change their layer counts, hidden dimensions, and vocabularies.
*   **Pretrained Profiles (e.g., `smollm2_135m`):** These are specifically designed to bridge external models into our local pipeline. 

**⚠️ The Golden Rule for Pretrained Profiles:**
Because pretrained profiles must exactly match the mathematical shape of the Hugging Face model they are importing, **you cannot change their structural architecture** (such as the number of layers, attention heads, or hidden dimensions). If you do, the downloaded Hugging Face weights will not fit! 

However, you are completely free to tweak their *behavioral* settings, such as modifying the generation temperature, or adjusting the batch size and learning rate for the SFT phase.

### 1.1 Inspect the Profiles

Profiles are the source of truth for model size, tokenizer, checkpoint paths, HF artifact locations, generation defaults, and SFT training config.

In [ ]:
from dustylm.config import get_profile, list_profiles

for name in ["dusty8m", "smollm2_135m", "smollm2_360m", "sft_smollm2_360m"]:
    profile = get_profile(name)
    model = profile.model
    print("=" * 80)
    print("profile:", profile.name)
    print("family:", model.family)
    print("layers:", model.num_layers)
    print("hidden dim:", model.embed_dim)
    print("heads:", model.num_heads, "query /", model.num_kv_heads, "kv")
    print("vocab:", model.vocab_size)
    print("tokenizer:", model.tokenizer.path_or_name)
    if profile.generation:
        print("checkpoint:", profile.generation.checkpoint_path)
    if profile.hf_artifacts:
        print("hf repo:", profile.hf_artifacts.repo_id)
        print("hf weights cache:", profile.hf_artifacts.local_weights_path)

---

## 2. Scaling Up: Importing External Weights

The `dusty8m` profile learned vocabulary, syntax, and world patterns entirely from our tiny local dataset. While this is fantastic for education because every step is visible, its capability is naturally limited.

Models like SmolLM2 solve this problem because they have already been trained on a massive corpus of human text using thousands of GPU hours. By importing these weights, your starting point is a model that already natively understands English and complex logic. DustyLM effortlessly converts these Hugging Face weights into our local module layout.

**Crucial Tokenizer Note:** Do not retrain the tokenizer for pretrained profiles! Pretrained weights are inextricably linked to the specific tokenizer used during their original training. The SmolLM2 profile already points to the correct, frozen tokenizer at `artifacts/tokenizers/smollm2_tokenizer.json`.

### 2.1 Download and Convert a Base Checkpoint

`dustylm.artifacts download --convert` does two things:

1. Downloads Hugging Face `model.safetensors` into `artifacts/hf/` and the tokenizer into `artifacts/tokenizers/`.
2. Converts the HF key names into a DustyLM `.pt` checkpoint under `artifacts/checkpoints/`.

The 135M profile is smaller and better for a first notebook run. The 360M profile is more capable but needs more memory and disk.

In [ ]:
PROFILE = "smollm2_135m"  # Change to "smollm2_360m" when you have enough memory/disk.

!uv run python -m dustylm.artifacts download --profile {PROFILE} --convert

## 3. Confirm Artifact Paths

After conversion, generation reads the `.pt` checkpoint from `artifacts/checkpoints/`.

In [ ]:
from pathlib import Path
from dustylm.config import get_profile

profile = get_profile(PROFILE)
paths = [
    profile.hf_artifacts.local_weights_path,
    profile.hf_artifacts.local_tokenizer_path,
    profile.generation.checkpoint_path,
]

for path in paths:
    path = Path(path)
    print(path, "OK" if path.exists() else "MISSING")

## 4. Generate From the Pretrained Base Model

Notice that this model doesn't have the "Dusty" personality yet. It is simply the raw SmolLM2 base model running through DustyLM's local generation stack.

In [ ]:
!uv run python -m dustylm.generate \
  --profile {PROFILE} \
  --prompt "A robot vacuum found a crumb under the couch and" \
  --temperature 0.8 \
  --top-p 0.9


## 5. How SFT Fits In

SFT is the alignment step: the pretrained SmolLM2 base already has broad language knowledge, and SFT teaches it the target response style or task behavior. The training mechanics are the same profile-driven path used elsewhere in DustyLM; the profile decides which tokenizer, dataset, initialization checkpoint, output checkpoint, and context length to use.

For SmolLM2 profiles, do not run `make dusty-tokenizer`. The tokenizer belongs to the pretrained model and is downloaded from Hugging Face. Your SFT dataset still needs to be tokenized for the SFT profile before training.

The repository currently ships `sft_smollm2_360m` for the larger profile. For a custom run with the 135m model, create or adjust an SFT profile that points at your SFT dataset and initializes from the converted `smollm2_135m` base checkpoint.

In [ ]:
# Example commands for a SmolLM2 SFT run after your SFT profile and dataset are ready.
# These are intentionally commented out because the dataset/profile choice is project-specific.

# !uv run python -m dustylm.data_prep --profile sft_smollm2_135m
# !uv run python -m dustylm.train --profile sft_smollm2_135m --epochs 1 --checkpoint-every-steps 100
# !uv run python -m dustylm.inference --profile sft_smollm2_135m

## 6. Practical Guidance

- Use `dusty8m` when you want to learn the full stack from scratch.
- Use `smollm2_135m` when you want a more capable pretrained base that is still relatively manageable.
- Use `smollm2_360m` when you have enough GPU memory and want stronger base capability.
- Keep SmolLM2 artifacts under `artifacts/hf/`, `artifacts/tokenizers/`, and `artifacts/checkpoints/`.
- Keep profile names and checkpoint paths explicit; that makes switching base models and SFT targets a config change instead of a code change.